# MODULE 34.11 — PRACTICAL AUDIO LIBRARIES 🎧

We've now studied audio preprocessing, feature extraction, speech recognition and ASR evaluation. It's time to understand the Python libraries that bring all these concepts together.

In this module, we'll focus on what each library does, why we need it, its important functions, and how to use it in an actual speech-processing project.

## Our Seven Audio Libraries

| Library | Primary purpose |
|---|---|
| `librosa` | Audio loading, analysis and feature extraction |
| `soundfile` | Reading and writing audio |
| `sounddevice` | Recording and playing audio |
| `scipy.signal` | Signal processing and filtering |
| `matplotlib` | Audio visualization |
| `torchaudio` | PyTorch audio processing |
| `transformers` | Pretrained ASR models |

---

## 1. Understanding the Audio Processing Ecosystem

These libraries are not direct replacements for one another. Each handles a different part of an audio-processing workflow.

### Audio processing workflow

```text
             Microphone / Audio File
                       |
                       v
                  Input Audio
                       |
                       v
                   SoundDevice
                 Recording / Playback
                       |
                       v
                    SoundFile
                  File Reading / Writing
                       |
                       v
                Librosa / Torchaudio
              Loading, Preprocessing
               and Feature Extraction
                       |
                       v
                  SciPy Signal
                  Filtering and DSP
                       |
                       v
                   Matplotlib
                   Visualization
                       |
                       v
                  Transformers
             Pretrained Speech Recognition
                       |
                       v
             Recognized Text / Application
```

This is one possible workflow, not a mandatory sequence. For example, Whisper can load and preprocess audio internally without requiring you to call Librosa first.

---

## 2. Librosa — Audio Loading, Analysis and Feature Extraction

<text-placeholder/>

**Priority: Must know**

![Librosa waveform and spectrogram visualization](https://images.openai.com/static-rsc-4/XU2HVc2jRxNcvqr8w6avKd112D1vdlwWZM3LQtKAv38s1bYMPzrmHBb6XmPmkRZ_Z0mqKAAJDEPn5OkW7zAZaC6kkYkfLj7h_LpIsrzP9muVcUzVMDub-bUcuFff1JThua7TcDgEWvs40X6-qz7xrkS29q8EEJPJCRwRG4tpVjM?purpose=inline)

### What is Librosa?

Librosa is a Python library for analysing audio signals and extracting meaningful audio features.

We have already used Librosa extensively in Modules 34.5, 34.6 and 34.7.

### What can Librosa do?

- Load audio files.
- Resample audio.
- Convert stereo audio to mono.
- Calculate STFT.
- Generate spectrograms.
- Extract MFCC features.
- Generate Mel spectrograms.
- Calculate RMS energy and ZCR.
- Perform time stretching and pitch shifting.
- Detect non-silent audio regions.

### Installation

```bash
pip install librosa
```

### A. Load audio

```python
import librosa

audio, sr = librosa.load(
    "speech.wav",
    sr=None,
    mono=True
)

print("Sample Rate:", sr)
print("Duration:", len(audio) / sr)
```

**Important:** `sr=None` preserves the original sample rate rather than automatically resampling to Librosa's default.

### B. Resample audio

```python
audio_16k = librosa.resample(
    audio,
    orig_sr=sr,
    target_sr=16000
)

print("New Sample Rate: 16000")
```

### C. Extract MFCC

```python
mfcc = librosa.feature.mfcc(
    y=audio_16k,
    sr=16000,
    n_mfcc=13
)

print("MFCC Shape:", mfcc.shape)
```

### D. Generate Mel spectrogram

```python
mel = librosa.feature.melspectrogram(
    y=audio_16k,
    sr=16000,
    n_mels=80
)

print("Mel Shape:", mel.shape)
```

### E. Time stretching

```python
stretched = librosa.effects.time_stretch(
    audio_16k,
    rate=0.8
)
```

This slows the audio while approximately preserving its pitch.

### When should you use Librosa?

Use Librosa when you want to:

- Explore audio signals.
- Extract traditional speech features.
- Build audio datasets.
- Visualize sound characteristics.
- Prepare inputs for machine learning models.

---

## 3. SoundFile — Reading and Writing Audio

![SoundFile and audio processing reference](https://images.openai.com/static-rsc-4/IS1KalNFRMGZoLiztMggQADZdGS4ngJaY7JlUrMYWQ5TwJ2vbjZ9ta3ZYDoJsXYNTFwIwbqFYvsZYyNoHwf5wJ6kkOsEMt0LVC7if974PnRX5tOhz61sy-6UD_-OpRKMIeJBs4FqvVbRQVX0luYO_yZBgZ_Y811mWoCnTSFpzYA?purpose=inline)

### What is SoundFile?

SoundFile provides audio file input/output using NumPy arrays. It is built around the `libsndfile` library.

### What can SoundFile do?

- Read WAV, FLAC and supported OGG files.
- Write audio files.
- Access audio metadata.
- Read audio in blocks.
- Save processed audio.

### Installation

```bash
pip install soundfile
```

### A. Read audio

```python
import soundfile as sf

audio, sr = sf.read("speech.wav")

print("Sample Rate:", sr)
print("Shape:", audio.shape)
```

### Important difference from Librosa

SoundFile returns multichannel audio in this shape:

```text
(number_of_samples, number_of_channels)
```

For example:

```text
(160000, 2)
```

This means:

- 160,000 samples.
- Two audio channels (stereo).

Librosa's multichannel output uses channel-first orientation when `mono=False`.

### B. Write audio

```python
sf.write(
    "processed_audio.wav",
    audio,
    sr
)
```

This saves the audio using its original sample rate.

### C. Save as 16-bit PCM WAV

```python
sf.write(
    "output.wav",
    audio,
    sr,
    subtype="PCM_16"
)
```

### D. Read audio in blocks

Useful when working with large recordings that you don't want to load entirely into memory.

```python
for block in sf.blocks(
    "long_recording.wav",
    blocksize=16000
):
    print(block.shape)
```

Each block contains a portion of the audio.

### When should you use SoundFile?

Use it when you need reliable audio file reading, saving, format conversion or block-based access.

---

## 4. SoundDevice — Recording and Playing Audio

![Python audio recording reference](https://images.openai.com/static-rsc-4/qumvLHEIKWdvud5JljqSLn3CnBMOesiyfmYVNOIXzI56s80LQzeecXHjEok4iACPz0iY3Jv9EWqVMHJo1rnOBTIsSlIzOHjYOGv99YXOV-M_bjiKK1nT5ZAFknyUsQHPDkheFY8hfHRz-yKPOs7wLXIypcRq0OzjqhAL_L2SH2k?purpose=inline)

### What is SoundDevice?

SoundDevice provides access to microphone and speaker devices through PortAudio.

So far, we've loaded audio files that already exist.

But what if we want to record our own voice directly from the microphone?

That's where SoundDevice becomes useful.

### Installation

```bash
pip install sounddevice
```

### A. Record audio

```python
import sounddevice as sd
import numpy as np

sample_rate = 16000
duration = 5

print("Recording started...")

recording = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="float32"
)

sd.wait()

print("Recording completed!")
print("Shape:", recording.shape)
```

### Understanding the parameters

| Parameter | Meaning |
|---|---|
| `duration` | Recording duration in seconds |
| `sample_rate` | Samples captured per second |
| `channels=1` | Mono recording |
| `dtype` | Data type of the audio |
| `sd.wait()` | Wait until recording finishes |

### B. Save the recording

```python
import soundfile as sf

sf.write(
    "my_voice.wav",
    recording,
    sample_rate
)
```

### C. Play audio

```python
sd.play(
    recording,
    samplerate=sample_rate
)

sd.wait()
```

### D. Check available audio devices

```python
print(sd.query_devices())
```

This lists available microphones and audio output devices.

### Practical applications

- Voice recording applications.
- Real-time speech recognition.
- Voice assistants.
- Live audio monitoring.
- Speech dataset collection.

### Important

The requested sample rate must be supported by your selected microphone. If recording fails, inspect the device's supported sample rates and channel count. Also check Windows microphone permissions.

---

## 5. SciPy Signal — Digital Signal Processing

### What is SciPy Signal?

The `scipy.signal` module provides mathematical and computational tools for analysing and processing signals.

This library is especially useful when we want more control over digital signal processing.

### What can `scipy.signal` do?

- Low-pass filtering.
- High-pass filtering.
- Band-pass filtering.
- Band-stop filtering.
- Signal resampling.
- Filter design.
- Convolution.
- Spectral analysis.

### Installation

```bash
pip install scipy
```

### A. Butterworth filter

A Butterworth filter is a type of digital filter designed to have a maximally flat frequency response in its passband.

We can use it to remove unwanted frequency components.

![Butterworth filter frequency response reference](https://images.openai.com/static-rsc-4/ecUbhRFOgXBuxc7WWIMbBKvlTup69iAy0odkP_NQ2SZXTE4_ZU3UEVKWIYinMdnUHhkaE9EuxLRctvtDHnAxoVPrPud3MlVfJ1Ba59Y_ts-6FH8PAAzeQmv9_Y3UPnxoeAExYqDyFhvsgwK7OoQSxC3Mzy5zu4JDAXsXFIpnAss?purpose=inline)

### B. Apply a band-pass filter

Let's retain frequencies between 80 Hz and 3,400 Hz as a demonstration.

```python
from scipy.signal import butter, sosfiltfilt

low_cutoff = 80
high_cutoff = 3400

sos = butter(
    4,
    [low_cutoff, high_cutoff],
    btype="bandpass",
    fs=sample_rate,
    output="sos"
)

filtered_audio = sosfiltfilt(
    sos,
    recording
)
```

### Understanding the code

- `butter()` designs the Butterworth filter.
- `4` specifies the filter design order.
- `btype="bandpass"` selects a band-pass filter.
- `fs` specifies the sampling frequency.
- `output="sos"` produces second-order sections.
- `sosfiltfilt()` filters forward and backward, avoiding phase delay in the ideal interior response.

SciPy recommends second-order sections for numerical stability in general-purpose filtering.

### C. Save the filtered audio

```python
sf.write(
    "filtered_speech.wav",
    filtered_audio,
    sample_rate
)
```

### D. Compare the original and filtered audio

```python
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.subplot(2, 1, 1)
plt.plot(recording)
plt.title("Original Audio")

plt.subplot(2, 1, 2)
plt.plot(filtered_audio)
plt.title("Filtered Audio")

plt.tight_layout()
plt.show()
```

### Important

A band-pass filter does not automatically remove all background noise. Noise inside the selected frequency band remains, and aggressive filtering can remove useful speech information.

Also, `sosfiltfilt()` is an offline, forward-backward operation. For live streaming, use a causal filter such as `sosfilt()` with properly maintained filter state.

---

## 6. Matplotlib — Audio Visualization

![Audio waveform sampling reference](https://images.openai.com/static-rsc-4/uusStBM0jINhXIZOfUnCsv2sW8hzmc0d3dpU7Cy5JutdRanNz0mQE3UmqKBAtrkWuVXvBs0gl3Aoo30JzoewpBEAH31mVyVj6_ecdqLWqd5HaHrffPeejYRSIUv7pG5iHje-oKK5riPR6K-zWzNC4ZTSkAWWjjZ8Xv4AbpGfVDU?purpose=inline)

### What is Matplotlib?

Matplotlib helps us visualize audio signals and their extracted features.

We have already used Matplotlib to plot:

- Waveforms.
- Spectrograms.
- Mel spectrograms.
- MFCCs.
- RMS energy.

### Installation

```bash
pip install matplotlib
```

### A. Waveform visualization

```python
import matplotlib.pyplot as plt
import numpy as np

time = np.arange(len(recording)) / sample_rate

plt.figure(figsize=(14, 4))

plt.plot(time, recording)

plt.title("Recorded Speech")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.show()
```

### B. Frequency spectrum

```python
from scipy.fft import rfft, rfftfreq

spectrum = np.abs(rfft(recording[:, 0]))

frequencies = rfftfreq(
    len(recording),
    d=1/sample_rate
)

plt.figure(figsize=(14, 4))

plt.plot(frequencies, spectrum)

plt.title("Frequency Spectrum")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")

plt.show()
```

### C. Spectrogram

```python
plt.figure(figsize=(14, 5))

plt.specgram(
    recording[:, 0],
    NFFT=512,
    Fs=sample_rate,
    noverlap=384
)

plt.title("Speech Spectrogram")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency (Hz)")

plt.colorbar(label="Intensity")
plt.show()
```

### When should you use Matplotlib?

Whenever you need to inspect audio, compare recordings, understand noise, visualize extracted features or analyse model outputs.

---

## 7. TorchAudio — PyTorch Audio Processing

**Priority: Important for deep learning**

![TorchAudio documentation reference](https://images.openai.com/static-rsc-4/DlMutvaOYyz-2MyYZIRVCQeBvgCXzYm00Ur38jHHfImYp79p0pmH5pTO11Ye6fen8rJDB6sIhODErIjBAMybpOE_k93Js7R6vN_UlWxt9i7iynBW5oICr4J1iRGRHrtybHKjgvEGtGdceLEAjLmyL_SaWASVRMyed-bMmupIxqo?purpose=inline)

### What is TorchAudio?

TorchAudio provides audio-processing components designed to work with PyTorch tensors and deep-learning workflows.

So far, most of our audio-processing code has used NumPy arrays.

But when we train neural networks using PyTorch, we generally need audio represented as PyTorch tensors.

That's where TorchAudio becomes useful.

### What can TorchAudio do?

- Audio loading and saving.
- Resampling.
- Mel spectrogram extraction.
- MFCC extraction.
- Audio transformations.
- Dataset preparation.
- Speech-processing model components.

### Installation

```bash
pip install torchaudio
```

**Version compatibility matters:** TorchAudio must be installed with a compatible PyTorch build. For CUDA, use the matching PyTorch installation instructions rather than installing arbitrary versions independently.

**API note:** TorchAudio entered a maintenance phase beginning with version 2.8. Since version 2.9, its audio decoding and encoding functionality has moved to TorchCodec; `torchaudio.load()` and `torchaudio.save()` are convenience wrappers around it. New projects should be aware of this transition.

### A. Load audio as a PyTorch tensor

```python
import torch
import torchaudio

audio_tensor, sr = torchaudio.load(
    "speech.wav"
)

print("Audio Shape:", audio_tensor.shape)
print("Sample Rate:", sr)
print("Device:", audio_tensor.device)
```

Typically, the shape is:

```text
(channels, samples)
```

For example:

```text
torch.Size([1, 160000])
```

This means:

- One audio channel.
- 160,000 samples.

### B. Move audio to GPU

```python
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

audio_tensor = audio_tensor.to(device)

print("Audio Device:", audio_tensor.device)
```

Now the audio tensor is on your selected device.

### C. Resample audio

```python
resampler = torchaudio.transforms.Resample(
    orig_freq=sr,
    new_freq=16000
)

audio_16k = resampler(audio_tensor)

print("Resampled Shape:", audio_16k.shape)
```

### D. Generate Mel spectrogram

```python
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=80
).to(device)

mel_spec = mel_transform(audio_16k)

print("Mel Spectrogram Shape:", mel_spec.shape)
```

### E. Generate MFCC

```python
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=16000,
    n_mfcc=13
).to(device)

mfcc_features = mfcc_transform(audio_16k)

print("MFCC Shape:", mfcc_features.shape)
```

### Why is TorchAudio useful?

Imagine building a speech emotion recognition model using PyTorch.

Your pipeline might look like:

```text
Audio Dataset
      |
      v
  TorchAudio
      |
      v
 Waveform Tensor
      |
      v
 Mel Spectrogram
      |
      v
  PyTorch CNN
      |
      v
Emotion Classification
```

You can keep the audio transformations and neural network operations within the PyTorch ecosystem.

### CPU vs GPU

Simple audio operations may be faster on CPU because transferring small audio tensors to GPU has overhead. GPU becomes more useful when processing large batches or running computationally expensive neural networks.

---

## 8. Transformers — Pretrained ASR Models

**Priority: Must know**

![Hugging Face Whisper model reference](https://images.openai.com/static-rsc-4/ZRrT_TECxnTYWnuuR1_uvkWdalqwN2WVrdz7RODRSChW_mQYPRVVWLlYuoBkcY0p4AjMuUbEB0vcSL6LM90q2q8hMdM5RMuteBP4PdjbwO6ZEiRp1Ze9U4qJVTsSb0QWHIYEPPQz-iv0kcmZNXqWwzhUyKd-0mwaJM3B1n8Cc80?purpose=inline)

### What is Hugging Face Transformers?

Transformers provides pretrained models and tools for using and fine-tuning speech recognition systems.

This is where our previous Whisper knowledge becomes useful.

Instead of building an ASR model from scratch, we can load a pretrained model and use it to transcribe audio.

### What can Transformers do for speech?

- Automatic speech recognition.
- Audio classification.
- Speech translation.
- Speech feature extraction.
- Pretrained model loading.
- Model fine-tuning.

### Installation

```bash
pip install transformers torch accelerate
```

### A. Load Whisper using Transformers

We'll use the `pipeline` API.

```python
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-base",
    device=device
)
```

The pipeline loads the required processor and model components.

### B. Transcribe audio

```python
result = asr_pipeline(
    "speech.wav",
    generate_kwargs={
        "language": "english",
        "task": "transcribe"
    }
)

print("Recognized Text:")
print(result["text"])
```

### C. Transcribe Tamil audio

```python
result = asr_pipeline(
    "tamil_speech.wav",
    generate_kwargs={
        "language": "tamil",
        "task": "transcribe"
    }
)

print(result["text"])
```

Use a multilingual checkpoint for this experiment.

### D. Experiment with another pretrained model

You can also try:

```python
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device
)
```

Compare the results against `whisper-base`.

Hugging Face also provides lower-level APIs such as `AutoProcessor` and `AutoModelForCTC` for CTC-based ASR models. These are useful when you want direct control over preprocessing, inference and decoding.

### Why use Transformers instead of implementing everything manually?

| Manual implementation | Transformers |
|---|---|
| Build or configure architecture | Load pretrained architecture |
| Manage model weights | Download pretrained weights |
| Implement preprocessing | Use model-specific processor |
| Implement decoding | Use supported decoding tools |
| Train from scratch or fine-tune | Inference or fine-tuning |

**Important:** Transformers is not itself an ASR architecture. It is a framework that provides implementations of architectures and pretrained checkpoints.

---

## 9. Combining the Libraries — Mini Project

Now let's combine the libraries into one small application.

### Project: Record → Filter → Visualize → Transcribe

### Speech Processing Pipeline

1. **Record speech:** SoundDevice captures microphone audio.
2. **Save audio:** SoundFile writes the recording to WAV.
3. **Filter audio:** SciPy applies a band-pass filter.
4. **Visualize audio:** Matplotlib displays the waveform.
5. **Transcribe speech:** Hugging Face Transformers runs pretrained Whisper.
6. **Analyse features:** Librosa extracts MFCC and Mel spectrogram features.

### Complete implementation — Dependencies

Install the dependencies:

```bash
pip install librosa soundfile sounddevice scipy matplotlib numpy torch torchaudio transformers accelerate
```

### Implementation source supplied

The supplied source contains the following final implementation segment:

```python
# =====================================
# 7. EXTRACT MEL SPECTROGRAM
# =====================================

mel = librosa.feature.melspectrogram(
    y=filtered_audio,
    sr=SR,
    n_mels=80
)

mel_db = librosa.power_to_db(
    mel,
    ref=np.max
)

plt.figure(figsize=(12, 5))

librosa.display.specshow(
    mel_db,
    sr=SR,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()


# =====================================
# 8. SPEECH RECOGNITION
# =====================================

device = 0 if torch.cuda.is_available() else -1

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device=device
)

result = asr(
    "filtered.wav",
    generate_kwargs={
        "language": "english",
        "task": "transcribe"
    }
)

print("\nRecognized Speech:")
print(result["text"])
```

**Practical note:** The band-pass filter here is for experimentation, not a guaranteed Whisper improvement. Compare transcription of `original.wav` and `filtered.wav`; filtering can sometimes remove useful speech information.

Also, the example uses offline recording and filtering. A real-time system would need streaming audio capture and a streaming-compatible recognition strategy.

---

## 10. Which Library Should You Use?

This is the practical decision-making table.

| Your requirement | Recommended library |
|---|---|
| Load audio and extract MFCC | Librosa |
| Save processed audio | SoundFile |
| Record your voice | SoundDevice |
| Remove selected frequency components | SciPy Signal |
| Plot waveform and spectrogram | Matplotlib |
| Prepare PyTorch audio tensors | TorchAudio |
| Run pretrained Whisper | Transformers |

### Important differences

#### Librosa vs TorchAudio

| Librosa | TorchAudio |
|---|---|
| NumPy-oriented | PyTorch-oriented |
| Excellent for traditional feature extraction | Designed for PyTorch audio workflows |
| Convenient audio analysis | Tensor-based transformations |
| CPU-oriented common workflows | Can integrate with GPU-based pipelines |

#### SoundFile vs SoundDevice

| SoundFile | SoundDevice |
|---|---|
| Works with existing audio files | Works with audio input/output devices |
| Read and write files | Record and play sound |
| File-based processing | Real-time audio interaction |

#### Transformers vs TorchAudio

| Transformers | TorchAudio |
|---|---|
| Pretrained ASR model implementations | Audio processing components |
| Speech recognition and translation models | Audio transforms and tensor operations |
| Model inference and fine-tuning | Audio data preparation |

---

## 11. Final Revision

### Your audio library toolkit

| Category | Library | Main responsibility |
|---|---|---|
| Analysis | Librosa | MFCC, Mel, RMS, ZCR, STFT |
| File I/O | SoundFile | Read, write and save audio |
| Recording | SoundDevice | Microphone and speakers |
| DSP | SciPy Signal | Filters and signal processing |
| Visualization | Matplotlib | Waveforms and spectrograms |
| Deep Learning | TorchAudio | PyTorch audio tensors and transforms |
| Pretrained AI Models | Hugging Face Transformers | Whisper, Wav2Vec2 and other pretrained speech models |

### Official documentation

- [Librosa documentation](https://librosa.org/doc/latest/ioformats.html?utm_source=chatgpt.com)
- [SoundFile documentation](https://pypi.org/project/soundfile/?utm_source=chatgpt.com)
- [SoundDevice documentation](https://python-sounddevice.readthedocs.io/_/downloads/en/latest/pdf/?utm_source=chatgpt.com)
- [SciPy Signal documentation](https://docs.scipy.org/doc/scipy/reference/signal.html?utm_source=chatgpt.com)
- [TorchAudio documentation](https://docs.pytorch.org/audio/?utm_source=chatgpt.com)
- [Hugging Face ASR documentation](https://huggingface.co/docs/transformers/tasks/asr?utm_source=chatgpt.com)

---

## What You Should Remember 🧠

1. Librosa is our primary traditional audio analysis library.
2. SoundFile handles audio file reading and writing.
3. SoundDevice connects Python to microphones and speakers.
4. SciPy provides digital signal-processing algorithms.
5. Matplotlib helps us visualize audio.
6. TorchAudio connects audio processing with PyTorch.
7. Transformers allows us to use pretrained speech recognition models.
8. Choose libraries according to the task; you don't need all seven in every project.

---